# 05 · Continuous (iteration-level) batching

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/05_continuous_batching.ipynb) &nbsp; 📖 Concept deck: [Batching](https://nilesh-salpe.github.io/visual-ai/inference/03-batching/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** an iteration-level scheduler (Orca, OSDI 2022) that admits new requests and retires finished ones at every decode step, compared with static batching.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def make_requests(n=40, seed=0, rate=0.5):
    """Requests arrive over time and need different numbers of output tokens."""
    r = np.random.default_rng(seed); t = 0.0; reqs = []
    for i in range(n):
        t += r.exponential(1 / rate)
        reqs.append({'id': i, 'arrival': int(t), 'out_len': int(r.integers(4, 60))})
    return reqs

def simulate_static(requests, max_batch):
    """Static batching: fixed groups (in arrival order) run until their LONGEST member finishes."""
    res, t, reqs = {}, 0, sorted(requests, key=lambda q: (q['arrival'], q['id']))
    for i in range(0, len(reqs), max_batch):
        g = reqs[i:i + max_batch]
        start = max(t, max(q['arrival'] for q in g))
        for q in g: res[q['id']] = (start, start + q['out_len'])
        t = start + max(q['out_len'] for q in g)
    return res

def metrics(requests, res):
    by = {q['id']: q for q in requests}
    span = max(f for _, f in res.values()) - min(q['arrival'] for q in requests)
    lat = [res[i][1] - by[i]['arrival'] for i in res]
    return {'throughput_tok_per_step': round(sum(q['out_len'] for q in requests) / span, 2), 'mean_latency': round(float(np.mean(lat)), 1), 'p90_latency': round(float(np.percentile(lat, 90)), 1)}
print('helpers ready: make_requests(), simulate_static(), metrics()')

## Task 1: Iteration-level scheduler

Simulate decode iterations `t = 0, 1, 2, …`, where each iteration produces one token for every running request:
- At the **start** of iteration `t`, admit waiting requests with `arrival <= t` in FIFO order (by `arrival`, then `id`) while fewer than `max_batch` are running.
- A request admitted at `t` with `out_len = L` occupies a slot for iterations `t … t+L-1` and **finishes at time `t + L`**. Its slot is free from iteration `t + L` onwards.
- If nothing is running and nothing has arrived yet, jump ahead to the next arrival.

Return `(results, batch_sizes)`, where `results[id] = (start, finish)` and `batch_sizes` lists the number of running requests in each iteration that ran.

In [ ]:
def simulate_continuous(requests, max_batch):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _ref_cont(requests, max_batch):
    q = collections.deque(sorted(requests, key=lambda r: (r['arrival'], r['id'])))
    running, res, sizes, t = [], {}, [], 0
    while q or running:
        running = [r for r in running if res[r['id']][1] > t]
        while q and q[0]['arrival'] <= t and len(running) < max_batch:
            r = q.popleft(); res[r['id']] = (t, t + r['out_len']); running.append(r)
        if not running:
            if q: t = q[0]['arrival']
            continue
        sizes.append(len(running)); t += 1
    return res, sizes

def _test_simulate_continuous(f):
    for seed, B in [(0, 4), (1, 8), (2, 1)]:
        reqs = make_requests(30, seed)
        res, sizes = f(reqs, B)
        ref, rsizes = _ref_cont(reqs, B)
        assert set(res) == set(ref), "every request must appear in results"
        for q in reqs:
            s, e = res[q['id']]
            assert s >= q['arrival'], f"request {q['id']} started before it arrived"
            assert e - s == q['out_len'], f"request {q['id']}: finish - start must equal out_len"
        assert max(sizes) <= B, "batch exceeded max_batch"
        assert res == ref and sizes == rsizes, f"schedule differs from reference (seed={seed}, max_batch={B}): admit FIFO at the START of each iteration, free slots at t + out_len"

check("simulate_continuous", _test_simulate_continuous, simulate_continuous)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Iteration-level scheduler
def simulate_continuous_solution(requests, max_batch):
    queue = collections.deque(sorted(requests, key=lambda r: (r['arrival'], r['id'])))
    running, res, sizes, t = [], {}, [], 0
    while queue or running:
        running = [r for r in running if res[r['id']][1] > t]            # retire finished
        while queue and queue[0]['arrival'] <= t and len(running) < max_batch:
            r = queue.popleft(); res[r['id']] = (t, t + r['out_len']); running.append(r)
        if not running:
            if queue: t = queue[0]['arrival']                          # idle: jump to next arrival
            continue
        sizes.append(len(running)); t += 1                             # one decode iteration
    return res, sizes

check("simulate_continuous (reference)", _test_simulate_continuous, simulate_continuous_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
reqs = make_requests(80, seed=3, rate=0.6)
for B in (4, 8, 16):
    s = metrics(reqs, simulate_static(reqs, B))
    c = metrics(reqs, simulate_continuous_solution(reqs, B)[0])
    print(f"max_batch={B:2d}  static: {s}\n              continuous: {c}")

---
## Takeaways
- Static batching holds every slot until the batch's longest request finishes, which wastes GPU time and makes new arrivals wait.
- Iteration-level scheduling refills slots every step. Orca reported **36.9×** throughput over FasterTransformer at the same latency (GPT-3 175B).
- It is the default in modern servers, and it combines with paged KV (notebook 07) and chunked prefill (notebook 06).